# Autodata 3.45 — extractor cloud v4 (RAR modern)

**Remediere nouă:** `ERROR: Unsupported Method` la 81 fișiere este o problemă de suport al metodei RAR, nu dovadă de parolă. Folosim executabilul `7zz` din ultima versiune oficială Linux, nu p7zip vechi. Nu executăm niciun program din arhivă.

Rulează **Runtime → Run all** și autorizează contul Drive cu arhiva `Autodata__3.45.rar` (~2,28 GB). Dacă runtime-ul Colab anterior există încă și fișierul temporar e prezent, descărcarea se reia automat. În caz contrar începe de la zero. Rezultatul este salvat în Drive ca `Autodata_3_45_EXTRACTED/` cu inventar și pachete ZIP sub 190 MB. Notebookul nu este încă validat pe arhiva completă.

**Criteriu de succes:** `TRANSFER FINALIZAT` după executarea ultimei celule. La eroare, trimite doar ultimele linii din jurnal, nu parola sau tokenuri.


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth, os, re, sys, time, json, shutil, hashlib, subprocess, getpass
from pathlib import Path
from google.auth.transport.requests import AuthorizedSession
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

FILE_ID = "1lC8pgAMU4RMkcatQClkzNnABJZpsKxZE"
ROOT = Path("/content/autodata_v2")
ARCHIVE = ROOT / "Autodata__3.45.rar"
EXTRACT = ROOT / "extracted"
PACKS = ROOT / "packages"
OUTPUT_FOLDER_NAME = "Autodata_3_45_EXTRACTED"
CHUNK = 8 * 1024 * 1024
PACK_LIMIT = 180 * 1024 * 1024
ROOT.mkdir(parents=True, exist_ok=True)

credentials, project = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
service = build("drive", "v3", credentials=credentials, cache_discovery=False)
session = AuthorizedSession(credentials)
meta = service.files().get(fileId=FILE_ID, fields="id,name,mimeType,size,md5Checksum,capabilities(canDownload)").execute()
assert meta["name"] == ARCHIVE.name, f"Fișier nepotrivit: {meta['name']}"
assert meta["mimeType"] != "application/vnd.google-apps.folder"
if meta.get("capabilities", {}).get("canDownload") is False:
    raise PermissionError("Contul nu are drept de descărcare.")
EXPECTED = int(meta["size"])
print("Drive API conectat. Arhiva:", meta["name"], f"({EXPECTED/2**30:.2f} GiB)")
print("Spațiu disponibil în Colab:", f"{shutil.disk_usage('/content').free/2**30:.1f} GiB")
if shutil.disk_usage("/content").free + (ARCHIVE.stat().st_size if ARCHIVE.exists() else 0) < EXPECTED + 1024**3:
    raise RuntimeError("Colab nu are spațiu pentru fișierul RAR. Schimbă runtime-ul cu mai mult disk.")

In [ ]:
# V3 — Drive REST API: reluare exactă, răspunsuri HTTP diagnosticate, retry cu backoff.
# Păstrează /content/autodata_v2/Autodata__3.45.rar dacă există o copie parțială.
import requests, random
from google.auth.transport.requests import Request

ENDPOINT = f"https://www.googleapis.com/drive/v3/files/{FILE_ID}"
TRANSIENT_HTTP = {408, 429, 500, 502, 503, 504}
TRANSIENT_403 = {"rateLimitExceeded", "userRateLimitExceeded", "backendError", "sharingRateLimitExceeded"}
MAX_ATTEMPTS = 14

def error_details(response):
    try:
        obj = response.json()
        err = obj.get("error", {})
        reasons = [x.get("reason", "") for x in err.get("errors", [])]
        return reasons[0] if reasons else "unknown", err.get("message", "")[:300]
    except Exception:
        return "unknown", "Răspuns HTTP fără detalii JSON"

def retry_delay(attempt, response=None):
    hint = None if response is None else response.headers.get("Retry-After")
    if hint and hint.isdecimal():
        return min(max(int(hint), 2), 120)
    return min(2 ** min(attempt, 6) + random.random(), 65)

if not ARCHIVE.exists():
    ARCHIVE.touch()
current = ARCHIVE.stat().st_size
if current > EXPECTED:
    raise RuntimeError("Copia parțială este mai mare decât sursa. Oprește și verifică fișierul.")
print(f"RELUARE de la {current}/{EXPECTED} octeți ({current/EXPECTED:.1%}).")
print("Răspunsurile HTTP temporare vor fi reîncercate fără pierderea progresului.")
with ARCHIVE.open("ab") as output:
    while current < EXPECTED:
        end = min(current + CHUNK, EXPECTED) - 1
        completed = False
        last_error = ""
        for attempt in range(MAX_ATTEMPTS):
            wait = None
            try:
                with session.get(ENDPOINT, params={"alt": "media"},
                                 headers={"Range": f"bytes={current}-{end}"},
                                 stream=True, timeout=(30, 180)) as response:
                    code = response.status_code
                    if code == 206:
                        header = response.headers.get("Content-Range", "")
                        expected_prefix = f"bytes {current}-{end}/{EXPECTED}"
                        if header.strip() != expected_prefix:
                            raise RuntimeError(
                                f"Content-Range nu corespunde: {header!r}; așteptat {expected_prefix!r}")
                        data = b"".join(response.iter_content(chunk_size=1024 * 1024))
                        if len(data) != end - current + 1:
                            raise IOError(f"Răspuns incomplet: {len(data)} octeți")
                        output.write(data)
                        output.flush()
                        current += len(data)
                        completed = True
                    else:
                        reason, message = error_details(response)
                        last_error = f"HTTP {code}, reason={reason}, message={message}"
                        print(f"Răspuns Google Drive: {last_error}")
                        if code == 403:
                            if reason in {"downloadQuotaExceeded", "dailyLimitExceeded", "storageQuotaExceeded"}:
                                raise PermissionError(
                                    "Google Drive indică o limită de cotă: "+last_error+
                                    f". Copia parțială ({current}/{EXPECTED} octeți) rămâne în /content; "
                                    "poate fi reluată după resetarea cotei, dacă runtime-ul Colab mai există.")
                            if reason not in TRANSIENT_403:
                                raise PermissionError(
                                    "Google Drive a refuzat cererea; nu este o eroare de rețea temporară. "+last_error)
                        elif code == 401:
                            credentials.refresh(Request())
                        elif code == 200:
                            raise RuntimeError(
                                "Drive a ignorat antetul Range (HTTP 200). "
                                "Nu pot anexa în siguranță întregul fișier la copia parțială. "+last_error)
                        elif code not in TRANSIENT_HTTP:
                            raise RuntimeError("Eroare permanentă la Drive API: "+last_error)
                        wait = retry_delay(attempt, response)
            except PermissionError:
                raise
            except (requests.RequestException, IOError) as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                print(f"Întrerupere rețea, încercarea {attempt+1}/{MAX_ATTEMPTS}: {last_error[:140]}")
                wait = retry_delay(attempt)
            if completed:
                break
            if wait is not None and attempt + 1 < MAX_ATTEMPTS:
                print(f"Reîncerc de la octetul {current} peste {wait:.0f} secunde...")
                time.sleep(wait)
        if not completed:
            raise RuntimeError(
                f"Nu s-a putut descărca segmentul {current}-{end} după {MAX_ATTEMPTS} încercări. "
                f"Ultima eroare: {last_error}. "
                f"Copia parțială este păstrată ({current/EXPECTED:.1%}).")
        if current % (128*2**20) < CHUNK or current == EXPECTED:
            print(f"Descărcare: {current/EXPECTED:.1%}")
assert ARCHIVE.stat().st_size == EXPECTED, "Descărcarea nu este completă."
md5 = hashlib.md5()
sha = hashlib.sha256()
with ARCHIVE.open("rb") as inp:
    for block in iter(lambda: inp.read(16 * 1024 * 1024), b""):
        md5.update(block)
        sha.update(block)
if meta.get("md5Checksum") and meta["md5Checksum"].lower() != md5.hexdigest().lower():
    raise RuntimeError("MD5 diferit de valoarea din Google Drive. Copia este invalidă.")
print("DOWNLOAD_VERIFIED=PASS SHA256:", sha.hexdigest())

In [ ]:
# V4 — 7-Zip oficial pentru Linux (7zz), în locul p7zip 16.x, care poate afișa "Unsupported Method" la RAR.
# Această celulă reutilizează RAR-ul verificat și descărcat de celulele anterioare.
import tarfile, io, platform, stat
from pathlib import Path

if not ARCHIVE.exists() or ARCHIVE.stat().st_size != EXPECTED:
    raise RuntimeError("RAR incomplet. Rulează mai întâi celula de descărcare.")

TOOL_DIR=ROOT/"tools"
TOOL_DIR.mkdir(parents=True,exist_ok=True)
SEVENZIP=TOOL_DIR/"7zz"

def install_official_7zz():
    arch=platform.machine().lower()
    platform_label={"x86_64":"x64","amd64":"x64","aarch64":"arm64"}.get(arch)
    if not platform_label: raise RuntimeError("Arhitectură Linux neacceptată: "+arch)
    release_url="https://api.github.com/repos/ip7z/7zip/releases/latest"
    print("Obțin 7-Zip oficial de la ip7z/7zip...")
    meta_response=requests.get(release_url,timeout=45)
    meta_response.raise_for_status()
    release=meta_response.json()
    version=release.get("tag_name","")
    assets=release.get("assets",[])
    matching=[a for a in assets if a.get("name","").endswith(f"-linux-{platform_label}.tar.xz")]
    if not matching:
        # Versiune oficială de rezervă, aleasă pentru compatibilitate RAR5.
        asset_name=f"7z2602-linux-{platform_label}.tar.xz"
        url=f"https://github.com/ip7z/7zip/releases/download/26.02/{asset_name}"
        digest=None
        print("Nu găsesc arhiva Linux în ultima versiune; folosesc 7-Zip 26.02.")
    else:
        asset=matching[0]
        url=asset["browser_download_url"]
        digest=asset.get("digest")
        print("Versiune:",version,"Fișier:",asset["name"])
    response=requests.get(url,timeout=(30,180),stream=True)
    response.raise_for_status()
    payload=bytearray()
    for block in response.iter_content(chunk_size=1024*1024):
        if block:
            payload.extend(block)
            if len(payload)>40*1024*1024:
                raise RuntimeError("Descărcarea 7-Zip depășește dimensiunea permisă.")
    if digest and digest.startswith("sha256:"):
        actual=hashlib.sha256(payload).hexdigest()
        if actual.lower()!=digest.split(":",1)[1].lower():
            raise RuntimeError("SHA256 7-Zip nu corespunde cu cel oficial.")
    with tarfile.open(fileobj=io.BytesIO(payload),mode="r:xz") as tar:
        entry=next((m for m in tar.getmembers() if m.isfile() and Path(m.name).name=="7zz"),None)
        if not entry: raise RuntimeError("Pachetul oficial nu conține executabilul 7zz.")
        if entry.size>25*1024*1024: raise RuntimeError("Executabil 7zz excesiv de mare.")
        extracted=tar.extractfile(entry)
        if extracted is None: raise RuntimeError("Nu pot citi executabilul 7zz.")
        with SEVENZIP.open("wb") as dst:
            shutil.copyfileobj(extracted,dst)
    SEVENZIP.chmod(0o755)
    result=subprocess.run([str(SEVENZIP),"i"],capture_output=True,text=True,timeout=30)
    if result.returncode!=0 or "7-Zip" not in result.stdout:
        raise RuntimeError("Executabilul 7zz nu a trecut testul de pornire.")
    print("7-Zip oficial instalat:",result.stdout.splitlines()[0:2])
    print("7ZZ_SETUP=PASS")
if not SEVENZIP.exists():
    install_official_7zz()
else:
    print("Reutilizez 7zz instalat în acest runtime.")

RAR_PASSWORD=None
def sevenzip(action,password=None):
    args=[str(SEVENZIP),action,"-y"]
    if password: args.append("-p"+password)
    else: args.append("-p-")
    if action=="x": args.append("-o"+str(EXTRACT))
    args.append(str(ARCHIVE))
    logfile=ROOT/f"sevenzip_{action}.log"
    with logfile.open("w",encoding="utf-8",errors="replace") as log:
        proc=subprocess.run(args,stdin=subprocess.DEVNULL,stdout=log,stderr=subprocess.STDOUT,timeout=7200)
    return proc.returncode,logfile

print("Verific RAR cu 7zz modern; nu introducem parolă pentru 'Unsupported Method'.")
rc,log=sevenzip("t")
if rc!=0:
    tail=log.read_text(errors="replace")[-3000:]
    print("Testul RAR a raportat eroare:\n",tail)
    low=tail.lower()
    if "unsupported method" in low or "unsupported compression" in low:
        raise RuntimeError("Metoda de compresie este încă nesuportată de 7zz. Este necesar un extractor unrar compatibil, nu o parolă.")
    if any(term in low for term in ["wrong password","encrypted","password is incorrect","password required"]):
        if input("Arhiva este criptată. Introduci o parolă? Scrie DA: ").strip().upper()=="DA":
            RAR_PASSWORD=getpass.getpass("Parolă arhivă: ")
            rc,log=sevenzip("t",RAR_PASSWORD)
    if rc!=0:
        raise RuntimeError("RAR_TEST_FAIL: "+log.read_text(errors="replace")[-1800:])
print("RAR_TEST=PASS")

args=[str(SEVENZIP),"l","-slt"]
if RAR_PASSWORD: args.append("-p"+RAR_PASSWORD)
else: args.append("-p-")
args.append(str(ARCHIVE))
listing=subprocess.run(args,capture_output=True,text=True,errors="replace",timeout=1800)
if listing.returncode: raise RuntimeError("Listarea RAR a eșuat: "+listing.stderr[-1000:])
(ROOT/"listare_7z.txt").write_text(listing.stdout,encoding="utf-8")
parts=listing.stdout.split("----------",1)
file_section=parts[1] if len(parts)>1 else listing.stdout
estimated=sum(int(s) for s in re.findall(r"(?m)^Size = (\d+)$",file_section))
print("Dimensiune estimată extrasă: %.2f GiB"%(estimated/2**30))
if estimated>22*1024**3: raise RuntimeError("Arhivă prea mare după dezarhivare (>22 GiB).")
from pathlib import PurePosixPath
for name in re.findall(r"(?m)^Path = (.+)$",file_section):
    path_in_archive=name.replace("\\","/")
    if (path_in_archive.startswith("/") or re.match(r"^[A-Za-z]:",path_in_archive)
        or ".." in PurePosixPath(path_in_archive).parts):
        raise RuntimeError("Cale nesigură în RAR: "+name[:150])
if shutil.disk_usage("/content").free < estimated+500*1024**2:
    raise RuntimeError("Nu există destul spațiu pe Colab pentru extragere.")
EXTRACT.mkdir(parents=True,exist_ok=True)
rc,log=sevenzip("x",RAR_PASSWORD)
if rc:
    raise RuntimeError("RAR_EXTRACT_FAIL: "+log.read_text(errors="replace")[-1800:])
print("RAR_EXTRACT=PASS")

In [ ]:
# Pachetizare în ZIP-uri <190 MB și manifest: conectorul Drive poate citi fiecare pachet.
# Folosim ZIP_STORED pentru mărimi previzibile și pentru a evita recompresia inutilă.
import zipfile
PACKS.mkdir(exist_ok=True)
for old in PACKS.glob("autodata_part_*.zip"): old.unlink()
entries=[]
for p in sorted(EXTRACT.rglob("*")):
    if not p.is_file() or p.is_symlink():
        continue
    relative=p.relative_to(EXTRACT).as_posix()
    entries.append((p, relative, p.stat().st_size))
if not entries: raise RuntimeError("Nu s-a extras niciun fișier.")
if len(entries)>150000: raise RuntimeError("Număr prea mare de fișiere; e necesară altă strategie.")
packing=[]
i=0; used=0; z=None; packname=None
def next_pack():
    global i,used,z,packname
    if z: z.close()
    i+=1;used=0
    packname=f"autodata_part_{i:04d}.zip"
    z=zipfile.ZipFile(PACKS/packname,"w",compression=zipfile.ZIP_STORED,allowZip64=True)
    print("Construiesc:",packname)
try:
    next_pack()
    for source, relative, size in entries:
        # Fișierele mari sunt împărțite în segmente care încap într-un ZIP.
        if size == 0:
            if used + 4096 > PACK_LIMIT: next_pack()
            z.writestr(relative,b"");used+=4096
            packing.append({"path":relative,"size_bytes":0,"part":packname})
            continue
        chunks=[]
        with source.open("rb") as f:
            index=0
            while True:
                buf=f.read(min(64*1024*1024, size - index*64*1024*1024))
                if not buf: break
                name=relative if size<=64*1024*1024 else f"{relative}.chunk{index:04d}"
                if used + len(buf) + 4096 > PACK_LIMIT: next_pack()
                z.writestr(name,buf)
                used+=len(buf)+4096
                chunks.append({"package":packname,"entry":name,"bytes":len(buf)})
                index+=1
        packing.append({"path":relative,"size_bytes":size,"chunks":chunks})
finally:
    if z: z.close()
packages=sorted(PACKS.glob("autodata_part_*.zip"))
for package in packages:
    assert package.stat().st_size < 190*1024*1024, "Un pachet a depășit plafonul."
manifest={"archive":ARCHIVE.name,"original_bytes":EXPECTED,"sha256":sha.hexdigest(),"file_count":len(packing),"package_count":len(packages),"files":packing}
(ROOT/"INVENTAR.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
(ROOT/"INVENTAR.txt").write_text("\n".join(f"{x['size_bytes']:>14} {x['path']}" for x in packing))
print("FILES=",len(entries),"PACKAGES=",len(packages))

In [ ]:
# Upload direct prin Drive API, fără drive.mount(), cu retry și posibilitate de reluare la rerulare.
from googleapiclient.errors import HttpError
def api_call(request):
    for attempt in range(8):
        try: return request.execute(num_retries=4)
        except HttpError as e:
            status=e.resp.status
            if status not in (429,500,502,503,504) and not (status==403 and b"rateLimitExceeded" in e.content):
                raise
            time.sleep(min(3*2**attempt,60))
    raise RuntimeError("Google Drive API: prea multe erori la operație.")
def drive_query(q):
    return api_call(service.files().list(q=q,fields="nextPageToken,files(id,name,size,mimeType)",pageSize=100,spaces="drive")).get("files",[])
matches=drive_query("name = '"+OUTPUT_FOLDER_NAME+"' and mimeType = 'application/vnd.google-apps.folder' and trashed = false")
folder_id = matches[0]["id"] if matches else api_call(service.files().create(
    body={"name":OUTPUT_FOLDER_NAME,"mimeType":"application/vnd.google-apps.folder"},
    fields="id")).get("id")
print("Folder destinație:",f"https://drive.google.com/drive/folders/{folder_id}")
def upload_one(localfile):
    # Evită duplicatele și reîncărcarea fișierelor deja finalizate în Drive.
    fname=localfile.name.replace("'","\\'")
    existing=drive_query(f"name = '{fname}' and '{folder_id}' in parents and trashed = false")
    for item in existing:
        if int(item.get("size","-1"))==localfile.stat().st_size:
            print("Există deja:",localfile.name)
            return item["id"]
    media=MediaFileUpload(str(localfile),mimetype="application/zip" if localfile.suffix==".zip" else "application/octet-stream",
                          chunksize=8*1024*1024,resumable=True)
    task=service.files().create(body={"name":localfile.name,"parents":[folder_id]},media_body=media,fields="id,name,size")
    result=None
    while result is None:
        progress,result=task.next_chunk(num_retries=7)
    print("UPLOAD OK:",localfile.name)
    return result["id"]
for source in [ROOT/"INVENTAR.json", ROOT/"INVENTAR.txt",ROOT/"listare_7z.txt"]+packages:
    upload_one(source)
# Marcare succes după ce toate pachetele au fost încărcate.
done=ROOT/"TRANSFER_FINALIZAT.txt"
done.write_text(f"PASS\nArchive SHA256: {sha.hexdigest()}\nPachete: {len(packages)}\nFișiere: {len(entries)}\n")
upload_one(done)
print("\nTRANSFER FINALIZAT. Verifică în Google Drive folderul Autodata_3_45_EXTRACTED")
print(f"https://drive.google.com/drive/folders/{folder_id}")